In [4]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. LOAD BOTH SETS OF EMBEDDINGS ---
print("Loading embeddings...")
era5_dict = np.load("era5_cnn_512_trained.npy", allow_pickle=True).item()
pangu_dict = np.load("pangu_cnn_512_trained.npy", allow_pickle=True).item()

# Find events that we have both predictions and ground truths for
common_events = list(set(era5_dict.keys()).intersection(set(pangu_dict.keys())))
print(f"Found {len(common_events)} matching events to compare.\n")

# Align them into arrays
era5_vecs = np.array([era5_dict[e] for e in common_events])
pangu_vecs = np.array([pangu_dict[e] for e in common_events])

# =====================================================================
# USE CASE 1: DIRECT EVALUATION (Did Pangu predict THIS storm correctly?)
# =====================================================================
print("--- USE CASE 1: Pangu Prediction Accuracy ---")

# Calculate similarity between Prediction and Truth for the SAME event
# .diagonal() gets the pair-wise matches (Event 1 vs Event 1, Event 2 vs Event 2)
accuracy_scores = cosine_similarity(pangu_vecs, era5_vecs).diagonal()

# Print some stats
print(f"Average Prediction Accuracy (Cosine Sim): {np.mean(accuracy_scores):.4f}")
print(f"Best Predicted Event: {common_events[np.argmax(accuracy_scores)]} (Score: {np.max(accuracy_scores):.4f})")
print(f"Worst Predicted Event: {common_events[np.argmin(accuracy_scores)]} (Score: {np.min(accuracy_scores):.4f})\n")


# =====================================================================
# USE CASE 2: THE WEATHER SEARCH ENGINE 
# (Find historical ERA5 events that share characteristics with a Pangu Prediction)
# =====================================================================
print("--- USE CASE 2: Structural Event Retrieval ---")

# Let's pick a random Pangu prediction to use as our "Search Query"
target_idx = 0 
target_event_name = common_events[target_idx]
target_pangu_vector = pangu_vecs[target_idx].reshape(1, -1) # Reshape for sklearn

print(f"🔍 Searching ERA5 history for storms similar to Pangu Prediction: {target_event_name}")

# Compare this ONE prediction against ALL historical ERA5 events
search_similarities = cosine_similarity(target_pangu_vector, era5_vecs)[0]

# Get the indices of the top 5 highest scores
top_5_indices = np.argsort(search_similarities)[::-1][:5]

print("\n🏆 Top 5 Most Similar Historical Events:")
for rank, idx in enumerate(top_5_indices):
    event_name = common_events[idx]
    score = search_similarities[idx]
    
    # Add a note if it found its own ground truth
    note = " (Exact Match!)" if event_name == target_event_name else ""
    print(f"  {rank + 1}. {event_name}: {score:.4f}{note}")

Loading embeddings...
Found 237 matching events to compare.

--- USE CASE 1: Pangu Prediction Accuracy ---
Average Prediction Accuracy (Cosine Sim): 0.6774
Best Predicted Event: FL_2020_000011_MOZ (Score: 0.9182)
Worst Predicted Event: FL_2022_000379_MYS (Score: 0.1762)

--- USE CASE 2: Structural Event Retrieval ---
🔍 Searching ERA5 history for storms similar to Pangu Prediction: FL_2019_000052_SRB

🏆 Top 5 Most Similar Historical Events:
  1. FL_2019_000052_SRB: 0.7123 (Exact Match!)
  2. FL_2021_000097_BGD: 0.4319
  3. FL_2019_000094_GTM: 0.4151
  4. FL_2021_000078_GTM: 0.3828
  5. TC_2018_000113_PHL: 0.3751


In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- 1. CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#input_root = "/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth"
#save_path = "era5_simple_cnn_1024_baseline.npy"
#input_root = "/home/teoaivalis/floods_kg/reanalysis_data/pangu_predictions"
#save_path = "pangu_simple_cnn_1024_baseline.npy"
input_root = "/home/teoaivalis/floods_kg/reanalysis_data/graphcast_predictions"
save_path = "graphcast_simple_cnn_1024_baseline.npy"

# We use the same 22-variable structure as the ClimaX logic for consistency
surface_vars = ['2m_temperature', 'mean_sea_level_pressure']
level_vars = ['temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind', 'geopotential']
levels = [1000, 850, 500, 250]

# --- 2. ARCHITECTURE DEFINITION ---
class WeatherCNNEncoder(nn.Module):
    def __init__(self, in_channels=22, out_dim=1024):
        super(WeatherCNNEncoder, self).__init__()
        
        self.features = nn.Sequential(
            # Input: [Batch, 22, 8, 8]
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2), # Output: [128, 4, 4]
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2), # Output: [256, 2, 2]
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim) # Final vector: 1024
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        return x

# --- 3. DATA PROCESSING FUNCTION ---
def process_to_cube(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
    
    channels = []

    # A. Surface Data (from 1000hPa rows)
    surf_df = df[df['level'] == 1000].sort_values(['latitude', 'longitude'], ascending=[False, True])
    # Safety: ensure exactly 64 points
    if len(surf_df) > 64: surf_df = surf_df.iloc[:64]
        
    for v in surface_vars:
        if v in surf_df.columns:
            grid = surf_df[v].values.reshape(8, 8)
        else:
            # Fallback if specific surface name is missing
            fallback = 'temperature' if 'temp' in v else 'geopotential'
            grid = surf_df[fallback].values.reshape(8, 8)
        
        # Standardize
        grid = (grid - grid.mean()) / (grid.std() + 1e-6)
        channels.append(grid)

    # B. Pressure Level Data (5 vars * 4 levels)
    for lvl in levels:
        lvl_df = df[df['level'] == lvl].sort_values(['latitude', 'longitude'], ascending=[False, True])
        if len(lvl_df) > 64: lvl_df = lvl_df.iloc[:64]
            
        for v in level_vars:
            grid = lvl_df[v].values.reshape(8, 8)
            grid = (grid - grid.mean()) / (grid.std() + 1e-6)
            channels.append(grid)
            
    # Resulting shape: [1, 22, 8, 8]
    return torch.from_numpy(np.stack(channels)).float().unsqueeze(0).to(device)

# --- 4. INITIALIZE BASELINE MODEL ---
model = WeatherCNNEncoder(in_channels=22, out_dim=1024).to(device)
model.eval()

# --- 5. EXTRACTION LOOP ---
all_embeddings = {}
folders = [f for f in os.listdir(input_root) if os.path.isdir(os.path.join(input_root, f))]

print(f"📊 Processing {len(folders)} events through CNN Baseline...")

with torch.inference_mode():
    for folder_name in tqdm(folders):
        csv_path = os.path.join(input_root, folder_name, "center_GC_pred.csv")
        
        if os.path.exists(csv_path):
            try:
                # 1. Cube-ify
                x = process_to_cube(csv_path)
                
                # 2. Extract 1024-dim Vector
                embedding = model(x).detach().cpu().numpy().flatten()
                
                # 3. Store in dictionary with folder name
                all_embeddings[folder_name] = embedding
                
            except Exception as e:
                print(f"\n⚠️ Error in {folder_name}: {e}")

# --- 6. SAVE ---
np.save(save_path, all_embeddings)
print(f"\n✅ SUCCESS!")
print(f"Total CNN Embeddings saved: {len(all_embeddings)}")
print(f"File path: {os.path.abspath(save_path)}")

✅ Using device: cpu
📊 Processing 57 events through CNN Baseline...


  0%|          | 0/57 [00:00<?, ?it/s]

100%|██████████| 57/57 [00:11<00:00,  5.16it/s]


✅ SUCCESS!
Total CNN Embeddings saved: 57
File path: /home/teoaivalis/floods_kg/reanalysis_data/graphcast_simple_cnn_1024_baseline.npy


chenge the cnn to output 512-dim

In [10]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- 1. CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# UNCOMMENT the source you want to process:

# --- ERA5 ---
input_root = "/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth"
save_path = "era5_cnn_512_baseline.npy"

# --- Pangu ---
#input_root = "/home/teoaivalis/floods_kg/reanalysis_data/pangu_predictions"
#save_path = "pangu_cnn_512_baseline.npy"

# --- GraphCast ---
#input_root = "/home/teoaivalis/floods_kg/reanalysis_data/graphcast_predictions"
#save_path = "graphcast_cnn_512_baseline.npy"

# Variable structure
surface_vars = ['2m_temperature', 'mean_sea_level_pressure']
level_vars = ['temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind', 'geopotential']
levels = [1000, 850, 500, 250]

print(f"✅ Using device: {device}")

# --- 2. ARCHITECTURE DEFINITION (REVISED FOR 512 DIM) ---
class WeatherCNNEncoder(nn.Module):
    def __init__(self, in_channels=22, out_dim=512):
        super(WeatherCNNEncoder, self).__init__()
        
        self.features = nn.Sequential(
            # Input: [Batch, 22, 8, 8]
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2), # Output: [128, 4, 4]
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2), # Output: [256, 2, 2]
        )
        
        # The flattened size from the features above is 256 * 2 * 2 = 1024
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 512), # Compress 1024 spatial features to 512
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim) # Final embedding: 512
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        return x

# --- 3. DATA PROCESSING FUNCTION ---
def process_to_cube(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
    
    channels = []

    # A. Surface Data (from 1000hPa rows)
    surf_df = df[df['level'] == 1000].sort_values(['latitude', 'longitude'], ascending=[False, True])
    if len(surf_df) > 64: surf_df = surf_df.iloc[:64]
        
    for v in surface_vars:
        if v in surf_df.columns:
            grid = surf_df[v].values.reshape(8, 8)
        else:
            fallback = 'temperature' if 'temp' in v else 'geopotential'
            grid = surf_df[fallback].values.reshape(8, 8)
        
        # Local Normalization
        grid = (grid - grid.mean()) / (grid.std() + 1e-6)
        channels.append(grid)

    # B. Pressure Level Data (5 vars * 4 levels)
    for lvl in levels:
        lvl_df = df[df['level'] == lvl].sort_values(['latitude', 'longitude'], ascending=[False, True])
        if len(lvl_df) > 64: lvl_df = lvl_df.iloc[:64]
            
        for v in level_vars:
            grid = lvl_df[v].values.reshape(8, 8)
            grid = (grid - grid.mean()) / (grid.std() + 1e-6)
            channels.append(grid)
            
    return torch.from_numpy(np.stack(channels)).float().unsqueeze(0).to(device)

# --- 4. INITIALIZE MODEL (512-DIM) ---
model = WeatherCNNEncoder(in_channels=22, out_dim=512).to(device)
model.eval()

# --- 5. EXTRACTION LOOP ---
all_embeddings = {}
folders = [f for f in os.listdir(input_root) if os.path.isdir(os.path.join(input_root, f))]

print(f"📊 Processing {len(folders)} events through CNN Baseline (512-dim)...")

# Detect the correct CSV filename based on path
def get_csv_filename(path):
    if "era5" in path.lower(): return "center_ERA5_truth.csv"
    if "pangu" in path.lower(): return "center_Pangu_pred.csv"
    if "graphcast" in path.lower(): return "center_GC_pred.csv"
    return "center_GC_pred.csv"

target_csv = get_csv_filename(input_root)

with torch.inference_mode():
    for folder_name in tqdm(folders):
        csv_path = os.path.join(input_root, folder_name, target_csv)
        
        if os.path.exists(csv_path):
            try:
                x = process_to_cube(csv_path)
                embedding = model(x).detach().cpu().numpy().flatten()
                all_embeddings[folder_name] = embedding
            except Exception as e:
                print(f"\n⚠️ Error in {folder_name}: {e}")

# --- 6. SAVE ---
np.save(save_path, all_embeddings)
print(f"\n✅ SUCCESS!")
print(f"Total 512-dim CNN Embeddings saved: {len(all_embeddings)}")
print(f"File path: {os.path.abspath(save_path)}")

✅ Using device: cpu
📊 Processing 1242 events through CNN Baseline (512-dim)...


  0%|          | 0/1242 [00:00<?, ?it/s]

100%|██████████| 1242/1242 [05:06<00:00,  4.05it/s]


✅ SUCCESS!
Total 512-dim CNN Embeddings saved: 1242
File path: /home/teoaivalis/floods_kg/reanalysis_data/era5_cnn_512_baseline.npy


extract embeddings with 256-dim

In [13]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- 1. CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SET YOUR PATHS HERE
input_root = "/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth"
save_path = "era5_cnn_256_baseline.npy"

#input_root = "/home/teoaivalis/floods_kg/reanalysis_data/pangu_predictions"
#save_path = "pangu_cnn_256_baseline.npy"

#input_root = "/home/teoaivalis/floods_kg/reanalysis_data/graphcast_predictions"
#save_path = "graphcast_cnn_256_baseline.npy"

surface_vars = ['2m_temperature', 'mean_sea_level_pressure']
level_vars = ['temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind', 'geopotential']
levels = [1000, 850, 500, 250]

print(f"✅ Using device: {device}")

# --- 2. ARCHITECTURE DEFINITION (REVISED FOR 256 DIM) ---
class WeatherCNNEncoder(nn.Module):
    def __init__(self, in_channels=22, out_dim=256):
        super(WeatherCNNEncoder, self).__init__()
        
        self.features = nn.Sequential(
            # Input: [Batch, 22, 8, 8]
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2), # Output: [128, 4, 4]
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2), # Output: [256, 2, 2]
        )
        
        # The flattened size is 256 * 2 * 2 = 1024
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 512), 
            nn.ReLU(),
            nn.Dropout(0.1),
            # Final output layer now projects down to 256
            nn.Linear(512, out_dim) 
        )

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        return x

# --- 3. DATA PROCESSING FUNCTION ---
def process_to_cube(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
    
    channels = []

    # A. Surface Data
    surf_df = df[df['level'] == 1000].sort_values(['latitude', 'longitude'], ascending=[False, True])
    if len(surf_df) > 64: surf_df = surf_df.iloc[:64]
        
    for v in surface_vars:
        col = v if v in surf_df.columns else ('temperature' if 'temp' in v else 'geopotential')
        grid = surf_df[col].values.reshape(8, 8)
        # Local Normalization
        grid = (grid - grid.mean()) / (grid.std() + 1e-6)
        channels.append(grid)

    # B. Pressure Level Data
    for lvl in levels:
        lvl_df = df[df['level'] == lvl].sort_values(['latitude', 'longitude'], ascending=[False, True])
        if len(lvl_df) > 64: lvl_df = lvl_df.iloc[:64]
            
        for v in level_vars:
            grid = lvl_df[v].values.reshape(8, 8)
            grid = (grid - grid.mean()) / (grid.std() + 1e-6)
            channels.append(grid)
            
    return torch.from_numpy(np.stack(channels)).float().unsqueeze(0).to(device)

# --- 4. INITIALIZE MODEL ---
model = WeatherCNNEncoder(in_channels=22, out_dim=256).to(device)
model.eval()

# --- 5. EXTRACTION LOOP ---
all_embeddings = {}
folders = [f for f in os.listdir(input_root) if os.path.isdir(os.path.join(input_root, f))]

# Auto-detect target filename
def get_target_filename(path):
    p = path.lower()
    if "era5" in p: return "center_ERA5_truth.csv"
    if "pangu" in p: return "center_Pangu_pred.csv"
    if "graphcast" in p: return "center_GC_pred.csv"
    return "center_GC_pred.csv"

target_csv = get_target_filename(input_root)

print(f"📊 Processing {len(folders)} events through 256-dim CNN...")

with torch.inference_mode():
    for folder_name in tqdm(folders):
        csv_path = os.path.join(input_root, folder_name, target_csv)
        
        if os.path.exists(csv_path):
            try:
                x = process_to_cube(csv_path)
                embedding = model(x).detach().cpu().numpy().flatten()
                all_embeddings[folder_name] = embedding
            except Exception as e:
                print(f"\n⚠️ Error in {folder_name}: {e}")

# --- 6. SAVE ---
np.save(save_path, all_embeddings)
print(f"\n✅ SUCCESS! saved to {save_path}")

✅ Using device: cpu
📊 Processing 1242 events through 256-dim CNN...


  0%|          | 0/1242 [00:00<?, ?it/s]

100%|██████████| 1242/1242 [06:06<00:00,  3.39it/s]


✅ SUCCESS! saved to era5_cnn_256_baseline.npy
